# Bài tập Buổi 5 — Pipeline Machine Learning: EDA & Tiền xử lý trên Titanic

### Sinh viên thực hiện
### Mai Gia Bảo - 24520163

**Khóa học hè 2026 — Python & Machine Learning · ML IoT Lab, HCMUT**

---

## Bối cảnh

Bạn vừa nhận được dataset **Titanic** — danh sách hành khách và việc họ có sống sót sau thảm họa hay không.
Nhiệm vụ của bạn **không phải** huấn luyện mô hình, mà là hoàn thành **hai bước đầu và quan trọng nhất** của một dự án Machine Learning:

> **Khám phá dữ liệu (EDA) → Tiền xử lý dữ liệu.**

Đây là phần chiếm ~70–90% công sức thực tế của một dự án ML. Một mô hình mạnh không thể cứu một bộ dữ liệu kém chất lượng.

## Mục tiêu bài tập

Sau khi hoàn thành, bạn sẽ chứng minh được rằng mình có thể:

1. Thực hiện **EDA đầy đủ** trên một dataset thực tế: kiểm tra cấu trúc, giá trị thiếu, outlier, phân phối và tương quan.
2. **Trực quan hóa** dữ liệu và **rút ra nhận xét có căn cứ** (không chỉ vẽ hình cho đẹp).
3. Áp dụng **đúng kỹ thuật tiền xử lý** cho từng loại dữ liệu: xử lý missing, encoding, scaling.
4. Chia tập và xây pipeline tiền xử lý **không rò rỉ dữ liệu (data leakage)**.
5. Viết **nhận xét tổng hợp** về dữ liệu như một data analyst thực thụ.

## Yêu cầu nộp bài

- Hoàn thiện notebook này (điền vào tất cả các ô `# TODO` và các phần *"Trả lời:"*).
- Notebook phải **chạy được từ trên xuống dưới không lỗi** (Kernel → Restart & Run All).
- Nộp qua **GitHub**: tải repo mẫu → đưa lên repo cá nhân → làm bài và nộp trên đó.

## Tiêu chí chấm (10 điểm)

| Nội dung | Điểm |
|---|---|
| EDA đầy đủ (shape/info/missing/outlier) | 2.0 |
| Trực quan hóa + nhận xét cho mỗi biểu đồ | 2.0 |
| Xử lý missing & outlier hợp lý, có giải thích | 1.5 |
| Encoding & scaling đúng loại biến | 1.5 |
| Chia tập & tiền xử lý **không leakage** | 1.5 |
| Nhận xét tổng hợp về dữ liệu | 1.5 |

> **Lưu ý về liêm chính học thuật:** được tham khảo tài liệu, nhưng phải **tự viết code và tự hiểu**. Phần nhận xét phải là quan sát của chính bạn từ dữ liệu.

---


## 0. Chuẩn bị môi trường

Ô này đã viết sẵn — chạy để nạp thư viện. Nếu thiếu thư viện, cài bằng `pip install pandas numpy matplotlib seaborn scikit-learn`.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


## 1. Tải dữ liệu (đã cho)

Ô này đã viết sẵn. Dữ liệu được tải từ `seaborn`, có fallback tải từ Internet nếu cần.

In [3]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
## Task 1 — Loại bỏ cột rò rỉ nhãn (data leakage) và cột dư thừa

### Mục đích
Trên slide đã học: **data leakage** là khi thông tin không được phép "rò" vào mô hình, khiến kết quả đẹp trên giấy nhưng vô dụng thực tế. Bản Titanic của seaborn chứa sẵn nhiều cột **rò rỉ nhãn** hoặc **trùng lặp**:

- `alive` (yes/no) — chính là `survived` viết bằng chữ ⇒ **rò rỉ target trực tiếp**. Để lại là mô hình "gian lận".
- `who`, `adult_male`, `class` — được suy ra từ `sex`, `age`, `pclass` (trùng thông tin).
- `deck` — thiếu quá nhiều (~77%).
- `embark_town` — trùng `embarked`; `alone` — suy ra từ `sibsp` + `parch`.

### Yêu cầu
1. In ra danh sách cột và tỷ lệ missing của **toàn bộ** dataframe (để thấy `deck` thiếu bao nhiêu).
2. Loại bỏ các cột rò rỉ / dư thừa ở trên, chỉ giữ lại:
   `survived, pclass, sex, age, sibsp, parch, fare, embarked`.
3. **Trả lời** (markdown ô dưới): vì sao để lại cột `alive` sẽ khiến mô hình đạt accuracy ~100% mà không thực sự học được gì?

### Gợi ý
- `df.isnull().mean()` cho tỷ lệ thiếu theo cột.
- `df.drop(columns=[...])` để bỏ cột.

In [ ]:
# TODO 1a: in tỷ lệ missing của tất cả các cột
print(df.isnull().mean())
# TODO 1b: bỏ các cột rò rỉ/dư thừa, gán lại vào biến df
# bỏ hết cột rò rỉ nhãn (alive) và cột trùng thông tin (who, adult_male, class, deck, embark_town, alone)
leaky = ['alive', 'who', 'adult_male', 'class', 'deck', 'embark_town', 'alone']   # chỉ giữ 8 cột gốc
df = df.drop(columns=leaky, errors="ignore")

print("Các cột còn lại:", list(df.columns))

In [5]:
df.isnull().mean()

survived       0.000000
pclass         0.000000
sex            0.000000
age            0.198653
sibsp          0.000000
parch          0.000000
fare           0.000000
embarked       0.002245
class          0.000000
who            0.000000
adult_male     0.000000
deck           0.772166
embark_town    0.002245
alone          0.000000
dtype: float64

**Trả lời 1c (vì sao `alive` gây rò rỉ target):**

*(viết tại đây...)*
Vì `alive` tương đương với cột target là `survived`. Nếu để model học được cái này thì nó sẽ tự suy diễn là nếu khách hàng nào có `alive` = `yes` thì nó sẽ dự đoán là khách đó còn sống luôn. Bỏ qua mọi feature khác nên phải xóa cột này đi

---
## Task 2 — Quan sát tổng quan

### Mục đích
Trước khi phân tích sâu, phải nắm được "hình dạng" của dữ liệu: bao nhiêu mẫu, bao nhiêu đặc trưng, kiểu dữ liệu từng cột, và thống kê cơ bản. Đây là bước đầu tiên của mọi EDA.

### Yêu cầu
1. In **số dòng và số cột**; nêu rõ đâu là **biến mục tiêu (target)**.
2. Dùng `df.info()` để xem kiểu dữ liệu và số giá trị non-null.
3. Dùng `df.describe()` cho biến số và `df.describe(include="object")` (hoặc `"category"`) cho biến phân loại.
4. **Trả lời:** cột nào là biến **số**, cột nào là biến **phân loại**?

In [13]:
# TODO 2: shape, info, describe
##df.shape()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(4)
memory usage: 73.7+ KB


,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [ ]:
print("Các biến phần loại object:")
print(df.describe(include="object"))
print("Các biến phần loại category:")
# sau khi Task 1 bỏ hết cột dạng category (class, deck), df có thể không còn cột category
if df.select_dtypes("category").shape[1] > 0:
    print(df.describe(include="category"))
else:
    print("(Đã bỏ các cột dạng category ở Task 1 nên không còn cột category để mô tả)")

**Trả lời 2 (biến số vs biến phân loại):**

*(viết tại đây...)*
- Biến số là các cột có thể được in ra trong phần df.info() như `survived`	`pclass`	`age`	`sibsp`	`parch`	`fare`.

- Biến phân loại là các cột được in ra bởi `df.describe(include="object")` hoặc `(include="category")`

---
## Task 3 — Missing Value: thống kê & đề xuất cách xử lý

### Mục đích
Mô hình học máy **không nhận trực tiếp giá trị NaN**. Nhưng cách xử lý phụ thuộc **tỷ lệ thiếu** và **vai trò của cột** — không có một cách đúng cho mọi trường hợp.

### Yêu cầu
1. Lập bảng: mỗi cột còn missing → **số lượng** và **phần trăm** thiếu.
2. Với **từng cột** còn thiếu, **đề xuất** cách xử lý và **giải thích ngắn gọn** (xóa / điền mean / điền median / điền mode / KNN...).

### Gợi ý
- Nhắc lại từ slide: `median` bền vững hơn `mean` khi có outlier; cột thiếu quá nhiều (>~60–70%) thường nên **bỏ**; biến phân loại thường điền **mode**.

In [ ]:
# TODO 3: bảng missing (count + %)
missing_count = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
bang_missing = pd.DataFrame({"so_luong_thieu": missing_count, "phan_tram_thieu": missing_pct})
# chỉ giữ các cột thực sự còn thiếu, sắp xếp giảm dần
bang_missing = bang_missing[bang_missing["so_luong_thieu"] > 0].sort_values("so_luong_thieu", ascending=False)
print(bang_missing)

**Trả lời 3 (đề xuất xử lý cho từng cột thiếu):**

| Cột | % thiếu | Cách xử lý đề xuất | Lý do |
|---|---|---|---|
| age | 19.87% | Điền bằng median | age có outlier và lệch nhẹ, median bền vững hơn mean, tỉ lệ thiếu vừa phải nên điền hợp lý hơn là xóa |
| embarked | 0.22% | Điền bằng mode (hay gặp nhất là S) | đây là biến phân loại, chỉ thiếu 2 dòng nên điền giá trị phổ biến nhất là an toàn và đơn giản |

---
## Task 4 — Phát hiện Outlier & **ra quyết định**

### Mục đích
Outlier có thể là **lỗi nhập liệu** (cần xử lý) hoặc **hiện tượng thật** (cần giữ). Phát hiện thôi chưa đủ — một analyst giỏi phải **quyết định** làm gì và giải thích được.

### Yêu cầu
1. Trên hai cột `age` và `fare`, đếm số outlier bằng **cả hai** phương pháp: **IQR** và **Z-score** (ngưỡng |z| > 3).
2. **Trả lời:** với các outlier của `fare`, bạn **giữ lại hay loại bỏ**? Vì sao? (gợi ý: nghĩ xem vé đắt bất thường là lỗi hay là vé hạng nhất có thật).

### Gợi ý
- IQR: outlier là điểm ngoài khoảng `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.
- Z-score: `from scipy import stats; np.abs(stats.zscore(series.dropna()))`.

In [ ]:
# TODO 4: đếm outlier theo IQR và Z-score cho 'age' và 'fare'
def dem_outlier_iqr(s):
    # trả về số lượng outlier theo IQR
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    can_duoi, can_tren = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < can_duoi) | (s > can_tren)).sum())

def dem_outlier_zscore(s, nguong=3.0):
    # trả về số lượng outlier theo Z-score
    s = s.dropna()
    z = np.abs(stats.zscore(s))
    return int((z > nguong).sum())

for col in ["age", "fare"]:
    print(f"{col}: IQR = {dem_outlier_iqr(df[col])} outlier, Z-score(|z|>3) = {dem_outlier_zscore(df[col])} outlier")

**Trả lời 4 (quyết định với outlier của `fare`):**

*(viết tại đây...)*

Với `fare`, tớ chọn GIỮ LẠI các outlier. Lý do là vé đắt bất thường không phải lỗi nhập liệu mà là vé hạng nhất có thật, ví dụ hành khách giàu ở khoang First class trả giá cao hơn hẳn. Những giá trị này mang thông tin thật về khả năng sống sót, vì người trả vé cao thường ở khoang tốt và được ưu tiên cứu hộ, nên bỏ đi sẽ mất tín hiệu quan trọng. Thay vì xóa, tớ dùng RobustScaler ở bước scaling để giảm ảnh hưởng của outlier mà vẫn giữ được dữ liệu.

---
## Task 5 — Trực quan hóa & nhận xét

### Mục đích
EDA là môn học về **nhìn** dữ liệu. Mỗi biểu đồ phải trả lời một câu hỏi và **đi kèm một nhận xét**. Vẽ mà không nhận xét thì không tính điểm.

### Yêu cầu — vẽ tối thiểu 4 loại biểu đồ, mỗi biểu đồ 1–2 câu nhận xét:
1. **Univariate — Histogram**: phân phối của `age` và `fare`. (Nhận xét: có lệch không? lệch trái hay phải?)
2. **Univariate — Boxplot**: `fare` theo nhóm `survived` hoặc `pclass`. (Nhận xét: outlier, trung vị.)
3. **Bivariate — Bar/Barplot**: **tỷ lệ sống sót** theo `sex` và theo `pclass`. (Nhận xét: nhóm nào sống nhiều hơn, chênh bao nhiêu %?)
4. **Multivariate — Heatmap**: ma trận tương quan giữa các biến số. (Nhận xét: cặp biến nào tương quan mạnh?)

### Gợi ý
- `sns.histplot`, `sns.boxplot`, `sns.barplot(data=df, x="sex", y="survived")` (barplot tự tính trung bình = tỷ lệ sống sót), `sns.heatmap(df.select_dtypes("number").corr(), annot=True)`.

In [ ]:
# TODO 5a: Histogram age & fare
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df["age"], kde=True, ax=axes[0], color="tab:blue")
axes[0].set_title("Phân phối tuổi (age)")
axes[0].set_xlabel("Tuổi")
axes[0].set_ylabel("Số hành khách")
sns.histplot(df["fare"], kde=True, ax=axes[1], color="tab:orange")
axes[1].set_title("Phân phối giá vé (fare)")
axes[1].set_xlabel("Giá vé")
axes[1].set_ylabel("Số hành khách")
plt.tight_layout()
plt.show()

In [ ]:
# TODO 5b: Boxplot fare theo survived hoặc pclass
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=df, x="survived", y="fare", ax=axes[0])
axes[0].set_title("Giá vé theo tình trạng sống sót")
axes[0].set_xlabel("survived (0 = mất, 1 = sống)")
axes[0].set_ylabel("Giá vé")
sns.boxplot(data=df, x="pclass", y="fare", ax=axes[1])
axes[1].set_title("Giá vé theo hạng vé")
axes[1].set_xlabel("pclass (hạng vé)")
axes[1].set_ylabel("Giá vé")
plt.tight_layout()
plt.show()

In [ ]:
# TODO 5c: Barplot tỷ lệ sống sót theo sex và pclass
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=df, x="sex", y="survived", ax=axes[0])
axes[0].set_title("Tỷ lệ sống sót theo giới tính")
axes[0].set_xlabel("Giới tính")
axes[0].set_ylabel("Tỷ lệ sống sót")
sns.barplot(data=df, x="pclass", y="survived", ax=axes[1])
axes[1].set_title("Tỷ lệ sống sót theo hạng vé")
axes[1].set_xlabel("pclass (hạng vé)")
axes[1].set_ylabel("Tỷ lệ sống sót")
plt.tight_layout()
plt.show()

# in số liệu cụ thể để nhận xét cho chính xác
print("Tỷ lệ sống sót theo giới tính:")
print(df.groupby("sex")["survived"].mean().round(3))
print("Tỷ lệ sống sót theo hạng vé:")
print(df.groupby("pclass")["survived"].mean().round(3))

In [ ]:
# TODO 5d: Heatmap correlation
plt.figure(figsize=(7, 5.5))
corr = df.select_dtypes("number").corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", square=True)
plt.title("Ma trận tương quan giữa các biến số")
plt.tight_layout()
plt.show()

**Nhận xét 5 (viết cho từng biểu đồ ở trên):**

- Histogram: `age` phân phối khá cân, tập trung nhiều ở khoảng 20 tới 40 tuổi và hơi lệch phải nhẹ, còn `fare` lệch phải rất mạnh, đa số vé rẻ dưới 50 nhưng có vài vé rất đắt kéo dài cái đuôi bên phải.
- Boxplot: người sống sót có giá vé trung vị cao hơn người không sống, và vé hạng 1 đắt hơn hẳn hạng 2 với hạng 3, `fare` có nhiều điểm outlier ở phía giá cao.
- Bar survival: nữ sống sót khoảng 74% còn nam chỉ khoảng 19%, chênh rất lớn, vé hạng 1 sống khoảng 63% rồi giảm dần xuống hạng 3 chỉ khoảng 24%, cho thấy giới tính và hạng vé ảnh hưởng mạnh tới khả năng sống.
- Heatmap: `pclass` tương quan âm với `survived` khoảng -0.34 và `fare` tương quan dương khoảng 0.26, nghĩa là hạng vé càng cao (số càng nhỏ) và trả tiền càng nhiều thì khả năng sống càng lớn, các biến số còn lại tương quan yếu.

---
## Task 6 — Chia tập **TRƯỚC** khi tiền xử lý (chống data leakage)

### Mục đích
Đây là điểm mấu chốt của buổi học. Mọi phép "học tham số" từ dữ liệu (median để điền, min/max/IQR để scale, danh mục để encode) **chỉ được học từ tập train**. Nếu học từ toàn bộ dữ liệu rồi mới chia, thông tin của tập test đã **rò rỉ** — điểm đánh giá sẽ ảo.

⇒ **Vì vậy phải chia tập TRƯỚC**, rồi mới xử lý.

### Yêu cầu
1. Tách `X` (đặc trưng) và `y` (`survived`).
2. Chia **train / validation / test** theo tỷ lệ khoảng **70 / 15 / 15**, có **`stratify=y`** để giữ nguyên tỷ lệ hai lớp.
3. In shape của 3 tập và **tỷ lệ sống sót** trong mỗi tập (để kiểm tra stratify hoạt động).

### Gợi ý
- Dùng `train_test_split` **hai lần**: lần 1 tách test (15%), lần 2 tách val từ phần còn lại.
- `stratify` nhận vào nhãn tương ứng ở mỗi lần chia.

In [ ]:
# TODO 6: chia train/val/test có stratify
X = df.drop(columns="survived")
y = df["survived"]

# X_tmp, X_test, y_tmp, y_test = train_test_split(...)
# X_train, X_val, y_train, y_val = train_test_split(...)
# lần 1: tách test 15%, còn lại 85%
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
# lần 2: tách val từ phần còn lại, 0.15/0.85 để val chiếm khoảng 15% tổng
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.15 / 0.85, stratify=y_tmp, random_state=42
)

# print("Train/Val/Test:", ...)
# in tỷ lệ survived từng tập
print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
print("Tỷ lệ sống sót train:", round(y_train.mean(), 3))
print("Tỷ lệ sống sót val:  ", round(y_val.mean(), 3))
print("Tỷ lệ sống sót test: ", round(y_test.mean(), 3))

---
## Task 7 — Xây pipeline tiền xử lý, **fit chỉ trên train**

### Mục đích
Gộp toàn bộ bước tiền xử lý vào một `ColumnTransformer` + `Pipeline`, `fit` **một lần trên `X_train`** rồi `transform` cho val/test. Đây là cách chuẩn để **đảm bảo không leakage** và tái sử dụng được.

### Yêu cầu
Xây `preprocess` gồm:

- **Biến số** (`age`, `sibsp`, `parch`, `fare`): `SimpleImputer(median)` → scaler (chọn `RobustScaler` vì `fare` có outlier, hoặc giải thích lựa chọn khác).
- **Biến phân loại** (`sex`, `embarked`): `SimpleImputer(most_frequent)` → `OneHotEncoder`.
- **Biến thứ tự** (`pclass`): giữ nguyên (`passthrough`) vì đã là số có thứ tự 1 < 2 < 3.

Sau đó: `fit` trên `X_train`, `transform` cho cả ba tập; in shape kết quả và tên cột sau biến đổi.

### Yêu cầu trả lời
- **Trả lời:** giải thích vì sao `fit` chỉ trên train (không phải trên toàn bộ dữ liệu) thì tránh được leakage.

### Gợi ý
- Khung `ColumnTransformer([... ("num", pipe_so, num_cols), ("cat", pipe_cat, cat_cols), ("ord", "passthrough", ord_cols)])`.
- `preprocess.get_feature_names_out()` để xem tên cột sau biến đổi.

In [ ]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

# TODO 7: xây pipeline cho biến số và biến phân loại
pipe_so  = Pipeline([
    # ("imputer", ...),
    # ("scaler",  ...),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
pipe_cat = Pipeline([
    # ("imputer", ...),
    # ("onehot",  ...),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    # ("num", pipe_so,  num_cols),
    # ("cat", pipe_cat, cat_cols),
    # ("ord", "passthrough", ord_cols),
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

# preprocess.fit(X_train)               # fit CHỈ trên train
# X_train_t = preprocess.transform(X_train)
# ... transform cho val, test
preprocess.fit(X_train)                 # fit CHỈ trên train để không rò rỉ dữ liệu
X_train_t = preprocess.transform(X_train)
X_val_t   = preprocess.transform(X_val)
X_test_t  = preprocess.transform(X_test)
print("Shape sau biến đổi:", X_train_t.shape, X_val_t.shape, X_test_t.shape)
print("Tên cột sau biến đổi:", list(preprocess.get_feature_names_out()))

**Trả lời 7 (vì sao fit chỉ trên train tránh leakage):**

*(viết tại đây...)*

Khi `fit` chỉ trên train, các tham số học được như median để điền thiếu, trung vị và IQR để scale, danh sách category để one-hot đều chỉ tính từ dữ liệu train. Nhờ vậy tập validation và test hoàn toàn mới lạ với pipeline, giống dữ liệu thật sẽ gặp sau này. Nếu tính các tham số đó trên toàn bộ dữ liệu rồi mới chia, thông tin của test đã lọt vào bước tiền xử lý, điểm đánh giá sẽ đẹp ảo và không phản ánh đúng năng lực thật của mô hình.

---
## Task 8 — Câu hỏi tư duy: chọn metric đánh giá

### Mục đích
Buổi học nhấn mạnh: **không có metric tốt nhất tuyệt đối** — phải chọn theo bài toán và mức mất cân bằng dữ liệu. Bài này không cần code, chỉ cần lập luận.

### Yêu cầu — trả lời ngắn gọn:
1. Biến mục tiêu `survived` có **mất cân bằng** không? (tính tỷ lệ hai lớp để trả lời).
2. Nếu chỉ nhìn **Accuracy**, có thể bị đánh lừa trong trường hợp nào?
3. Với bài toán Titanic, bạn sẽ ưu tiên metric nào (Accuracy / Precision / Recall / F1)? Vì sao?

In [ ]:
# TODO 8: tính tỷ lệ hai lớp của 'survived' để hỗ trợ trả lời
ty_le = df["survived"].value_counts(normalize=True).round(3)
print("Tỷ lệ hai lớp của survived:")
print(ty_le)
print(f"Lớp 0 (không sống): {ty_le[0]*100:.1f}%, lớp 1 (sống): {ty_le[1]*100:.1f}%")

**Trả lời 8:**

1. Có mất cân bằng nhẹ, lớp không sống chiếm khoảng 61.6% còn lớp sống khoảng 38.4%, chênh lệch không quá lớn nhưng vẫn đủ để cần lưu ý.
2. Chỉ nhìn Accuracy có thể bị đánh lừa, ví dụ một mô hình đoán tất cả đều chết đã đạt khoảng 61.6% accuracy mà thực ra không học được gì, vì nó bỏ sót toàn bộ người sống.
3. Với Titanic tớ ưu tiên F1 hoặc Recall cho lớp sống sót, vì mục tiêu là nhận ra được người sống, F1 cân bằng giữa Precision và Recall nên phản ánh tốt hơn Accuracy khi dữ liệu hơi lệch.

---
## Task 9 — Nhận xét tổng hợp về dữ liệu

### Mục đích
Khép lại toàn bộ EDA bằng một bản tóm tắt như một data analyst gửi cho đồng đội: **những gì đáng chú ý nhất** về bộ dữ liệu này.

### Yêu cầu — viết ít nhất 5 gạch đầu dòng, dựa trên **bằng chứng** (số liệu / biểu đồ) ở trên:
- Đặc trưng nào **tương quan mạnh nhất** với khả năng sống sót? (số liệu chứng minh)
- Cột nào **thiếu nhiều nhất** và bạn đã xử lý thế nào?
- Biến mục tiêu có **mất cân bằng** không? ảnh hưởng gì tới việc chọn metric?
- Đặc trưng nào cần **scaling**, đặc trưng nào cần **encoding**? vì sao?
- Một điều bạn thấy **bất ngờ / thú vị** trong dữ liệu.

**Nhận xét tổng hợp của bạn:**

1. Giới tính liên quan mạnh nhất tới sống sót, nữ sống khoảng 74% còn nam chỉ khoảng 19%, kế đến là hạng vé với `pclass` tương quan âm khoảng -0.34 với `survived`.
2. Trong 8 cột giữ lại, `age` thiếu nhiều nhất khoảng 19.9%, tớ xử lý bằng cách điền median vì age có outlier, riêng `deck` thiếu tới 77% nên đã bị loại ngay từ Task 1.
3. Biến mục tiêu `survived` mất cân bằng nhẹ khoảng 61.6% so với 38.4%, nên tớ ưu tiên F1 hoặc Recall thay vì chỉ nhìn Accuracy.
4. Các biến số như `age`, `fare`, `sibsp`, `parch` cần scaling vì thang đo rất khác nhau, còn `sex` và `embarked` cần encoding bằng one-hot, riêng `pclass` giữ nguyên vì đã là số có thứ tự.
5. Một điều thú vị là `fare` lệch phải rất mạnh và có nhiều vé đắt bất thường, nhưng đó là vé hạng nhất có thật chứ không phải lỗi, và chính nhóm này lại có tỷ lệ sống cao hơn hẳn.

---
## (Bonus — không bắt buộc) Thử thách nâng cao

Chọn **một** trong các hướng sau nếu bạn muốn thử sức:

1. **Feature engineering:** tạo đặc trưng mới `family_size = sibsp + parch + 1`, hoặc trích `title` (Mr/Mrs/Miss...) từ tên (nếu dùng bản có cột `name`). Kiểm tra tương quan với `survived`.
2. **So sánh scaler:** vẽ phân phối `fare` trước và sau khi áp `StandardScaler`, `MinMaxScaler`, `RobustScaler`. Nhận xét scaler nào phù hợp nhất với dữ liệu lệch + có outlier.
3. **Bẫy KNN:** thử `KNNImputer` để điền `age` **khi chưa scale** và **sau khi đã scale** `fare`. Quan sát kết quả có khác nhau không, và giải thích tại sao (gợi ý: khoảng cách Euclid bị chi phối bởi cột thang đo lớn).

In [ ]:
# (tùy chọn) code cho phần Bonus

# ===== Bonus 1: Feature engineering =====
df_bonus = df.copy()
df_bonus["family_size"] = df_bonus["sibsp"] + df_bonus["parch"] + 1
df_bonus["is_alone"] = (df_bonus["family_size"] == 1).astype(int)
df_bonus["is_child"] = (df_bonus["age"] < 16).astype(int)   # age thiếu cho ra False, coi như không phải trẻ em

print("Tỷ lệ sống theo family_size:")
print(df_bonus.groupby("family_size")["survived"].mean().round(3))
print("\nTỷ lệ sống theo is_alone (1 = đi một mình):")
print(df_bonus.groupby("is_alone")["survived"].mean().round(3))
print("\nTỷ lệ sống theo is_child (1 = trẻ em dưới 16 tuổi):")
print(df_bonus.groupby("is_child")["survived"].mean().round(3))
print("\nTương quan các đặc trưng mới với survived:")
print(df_bonus[["family_size", "is_alone", "is_child", "survived"]].corr()["survived"].round(3))

# ===== Bonus 2: So sánh scaler trên fare (fit chỉ trên train) =====
from sklearn.preprocessing import MinMaxScaler

fare_train = X_train[["fare"]].copy()
scalers = {"Gốc (chưa scale)": None, "StandardScaler": StandardScaler(),
           "MinMaxScaler": MinMaxScaler(), "RobustScaler": RobustScaler()}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (ten, sc) in zip(axes, scalers.items()):
    gt = fare_train["fare"].values if sc is None else sc.fit_transform(fare_train).ravel()
    sns.histplot(gt, kde=True, ax=ax, color="tab:green")
    ax.set_title(ten)
    ax.set_xlabel("fare")
plt.tight_layout()
plt.show()

# ===== Bonus 3: Bẫy KNNImputer khi CHƯA scale vs ĐÃ scale =====
from sklearn.impute import KNNImputer

so_cols = ["age", "sibsp", "parch", "fare"]
train_so = X_train[so_cols].copy()

# (a) KNN điền age khi chưa scale, fare thang đo lớn sẽ chi phối khoảng cách
age_chua_scale = KNNImputer(n_neighbors=5).fit_transform(train_so)[:, 0]

# (b) scale trước rồi mới KNN, các cột cân bằng hơn
scaler_tmp = RobustScaler()
train_scaled = scaler_tmp.fit_transform(train_so)
age_scaled = KNNImputer(n_neighbors=5).fit_transform(train_scaled)[:, 0]
# đưa age đã điền về lại thang gốc để so sánh cho công bằng
age_da_scale = age_scaled * scaler_tmp.scale_[0] + scaler_tmp.center_[0]

mask_thieu = X_train["age"].isna().values
print("Số dòng age bị thiếu trong train:", int(mask_thieu.sum()))
print("age điền KHI CHƯA scale (5 dòng thiếu đầu):", np.round(age_chua_scale[mask_thieu][:5], 2))
print("age điền SAU KHI scale (5 dòng thiếu đầu):", np.round(age_da_scale[mask_thieu][:5], 2))
print("Chênh lệch tuyệt đối trung bình:", round(float(np.mean(np.abs(age_chua_scale[mask_thieu] - age_da_scale[mask_thieu]))), 3))

**Nhận xét Bonus:**

- Feature engineering: người đi một mình (`is_alone` = 1) có tỷ lệ sống thấp hơn người đi cùng gia đình nhỏ, gia đình cỡ 2 tới 4 người sống cao nhất còn gia đình quá đông lại giảm, trẻ em (`is_child`) có tỷ lệ sống nhỉnh hơn, khớp với câu "phụ nữ và trẻ em trước". Các đặc trưng mới này nếu đưa vào mô hình thật sẽ giúp tăng điểm.
- So sánh scaler: cả ba scaler chỉ đổi thang đo chứ không làm hết lệch phải của `fare`, hình dạng phân phối vẫn giữ nguyên, trong đó RobustScaler ít bị outlier kéo giãn nhất vì nó dùng trung vị và IQR, nên hợp với `fare` lệch và nhiều outlier.
- Bẫy KNN: khi chưa scale, KNNImputer điền `age` bị chi phối bởi `fare` do `fare` có thang đo lớn hơn nhiều, sau khi scale các cột về cùng thang thì giá trị `age` điền ra khác đi, cho thấy cần scale trước khi tính khoảng cách.

---
## Bảng tự kiểm trước khi nộp

- [ ] Notebook chạy **Restart & Run All** không lỗi.
- [ ] Đã bỏ các cột rò rỉ/dư thừa (Task 1) và giải thích được vì sao.
- [ ] Mỗi biểu đồ (Task 5) đều có **nhận xét**.
- [ ] Đã **chia tập trước**, tiền xử lý **fit chỉ trên train** (Task 6–7).
- [ ] Đã trả lời tất cả các phần *"Trả lời:"*.
- [ ] Nhận xét tổng hợp (Task 9) có **ít nhất 5 ý** dựa trên bằng chứng.
- [ ] Đã push lên **repo cá nhân trên GitHub**.
